# 08_phase3_composition_scCODA.ipynb
Phase 3 — Compositional Analysis (scCODA)

**Why scCODA instead of the Phase 2 Kruskal-Wallis pass:** cell-type proportions are compositional data — they sum to 100% within each sample, so cell types aren't statistically independent (if one goes up, something else must go down). Standard tests (Kruskal-Wallis, Mann-Whitney) don't account for this. scCODA uses a Bayesian Dirichlet-Multinomial model built specifically for compositional data, modelling every cell type's change relative to one chosen reference type.

**Reference cell type choice matters** — it should be a population unlikely to be the one driving the biological difference under test, since scCODA's results are always interpreted relative to it. Chosen deliberately per comparison below, not defaulted.

In [2]:
pip install tf-keras

  Using cached tf_keras-2.21.0-py3-none-any.whl.metadata (1.8 kB)
Using cached tf_keras-2.21.0-py3-none-any.whl (1.7 MB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
# ----------------------------
# Cell 1 — Imports and paths
# ----------------------------
import gc
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
from pathlib import Path

try:
    from sccoda.util import cell_composition_data as scc_dat
    from sccoda.util import comp_ana as scc_ana
    print("sccoda imported successfully")
except ImportError:
    print("sccoda not installed. Run: pip install sccoda")
    raise

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
RESULTS_DIR = PROJECT_DIR / "results" / "phase3_composition_sccoda"
FIGURE_DIR = PROJECT_DIR / "figures" / "phase3_composition_sccoda"

for d in [RESULTS_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

EXCLUDE_TYPES = ["Unassigned (n=28, doublet/mixed-identity artefact)",
                  "Mixed/stromal-contaminated (CD8+fibroblast signal)"]

print("Setup complete")

C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\arviz\__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(


sccoda imported successfully
Setup complete


In [4]:
# ----------------------------
# Cell 2 — Build sample x cell_type count table (scCODA input format)
# FIX: condition_col cast to string to drop unused categorical levels —
# same protection applied here as in Cell 3, so this is safe for every
# comparison that uses this function, not just GSE114725.
# ----------------------------
def build_composition_anndata(adata_obs, sample_col, condition_col, exclude_types=EXCLUDE_TYPES):
    df = adata_obs[~adata_obs["cell_type"].isin(exclude_types)].copy()
    df[condition_col] = df[condition_col].astype(str)
    counts = df.groupby([sample_col, "cell_type"], observed=True).size().unstack(fill_value=0)

    condition_lookup = df.drop_duplicates(sample_col).set_index(sample_col)[condition_col]
    sample_meta = pd.DataFrame({condition_col: condition_lookup})

    comp_adata = scc_dat.from_pandas(counts, covariate_columns=[])
    comp_adata.obs[condition_col] = sample_meta.loc[comp_adata.obs_names, condition_col].values

    return comp_adata

print("Composition AnnData builder ready (categorical dtype fix applied)")

Composition AnnData builder ready (categorical dtype fix applied)


In [5]:
# ----------------------------
# Cell 3 — GSE114725: Tumour vs Normal
# FIX: obs1["tissue"] was a pandas Categorical retaining all 4 original
# levels (BLOOD, LYMPHNODE, NORMAL, TUMOR) even after filtering rows to
# just TUMOR/NORMAL — same categorical dtype bug already found and fixed
# in the pseudobulk DE notebook. This caused scCODA to fit a phantom
# tissue[T.LYMPHNODE] covariate with zero actual samples, which produced
# a degenerate fit (every single effect came back exactly 0.0 across
# every cell type — a strong sign of a broken model, not a genuine
# uniform null result). Cast to plain string immediately after filtering
# to drop the phantom category.
#
# Reference: T cells — the largest, most stable population, and not
# expected a priori to be THE cell type driving tumour/normal difference
# (macrophages are the stronger candidate for that, per DE/LIANA
# findings already established — using them as reference could bias
# results toward finding everything else "changed" relative to them).
# ----------------------------
adata1 = sc.read_h5ad(PROCESSED_DIR / "GSE114725_phase2_v2_annotated.h5ad", backed="r")
obs1 = adata1.obs[adata1.obs["tissue"].isin(["TUMOR", "NORMAL"])].copy()
obs1["tissue"] = obs1["tissue"].astype(str)  # drop unused categorical levels

comp_adata1 = build_composition_anndata(obs1, sample_col="patient", condition_col="tissue")
print(f"GSE114725 composition data: {comp_adata1.n_obs} samples x {comp_adata1.n_vars} cell types")
print(comp_adata1.obs)

model1 = scc_ana.CompositionalAnalysis(
    comp_adata1, formula="tissue", reference_cell_type="T cells"
)
result1 = model1.sample_hmc()
print("\n" + "="*60)
print("GSE114725: Tumour vs Normal — scCODA results")
print("="*60)
result1.summary()

GSE114725 composition data: 8 samples x 8 cell types
cell_type  tissue
patient          
BC1        NORMAL
BC2        NORMAL
BC3        NORMAL
BC4         TUMOR
BC5         TUMOR
BC6         TUMOR
BC7        NORMAL
BC8         TUMOR


100%|██████████| 20000/20000 [02:17<00:00, 145.26it/s]


MCMC sampling finished. (176.853 sec)
Acceptance rate: 53.1%

GSE114725: Tumour vs Normal — scCODA results
Compositional Analysis summary:

Data: 8 samples, 8 cell types
Reference index: 0
Formula: tissue

Intercepts:
                      Final Parameter  Expected Sample
Cell Type                                             
T cells                         1.247       700.241769
CD8/Effector T cells            1.333       763.127911
NK/Cytotoxic T cells            0.364       289.578281
B cells                        -0.132       176.342065
Macrophages                     1.159       641.254016
pDC                            -1.041        71.052970
Monocytes/DC                   -0.888        82.799801
Mast cells                     -0.692       100.728186


Effects:
                                      Final Parameter  Expected Sample  \
Covariate       Cell Type                                                
tissue[T.TUMOR] T cells                           0.0       700.241769   

In [6]:
# ----------------------------
# Cell 4 — GSE114725: extract and save credible effects
# scCODA reports "credible" effects (analogous to significant, but via
# Bayesian HDI/FDR rather than a classic p-value) — cell types whose
# change is confidently non-zero relative to the reference.
# ----------------------------
effects1 = result1.effect_df
effects1.to_csv(RESULTS_DIR / "GSE114725_scCODA_tumor_vs_normal_effects.csv")
print(effects1)

credible1 = result1.credible_effects()
print("\nCredible (confidently non-zero) effects:")
print(credible1)

                                      Final Parameter  HDI 3%  HDI 97%     SD  \
Covariate       Cell Type                                                       
tissue[T.TUMOR] T cells                           0.0   0.000    0.000  0.000   
                CD8/Effector T cells              0.0  -0.308    0.631  0.183   
                NK/Cytotoxic T cells              0.0  -0.450    0.681  0.204   
                B cells                           0.0  -0.662    0.519  0.202   
                Macrophages                       0.0  -0.220    0.866  0.257   
                pDC                               0.0  -0.575    0.665  0.225   
                Monocytes/DC                      0.0  -0.649    0.522  0.204   
                Mast cells                        0.0  -0.559    0.652  0.208   

                                      Inclusion probability  Expected Sample  \
Covariate       Cell Type                                                      
tissue[T.TUMOR] T cells      

In [7]:
# ----------------------------
# Cell 5 — GSE176078: pairwise subtype comparisons
# Reference: PVL (perivascular-like) — a structural/stromal population,
# not expected to be a primary driver of subtype-specific immune or
# epithelial differences (unlike CAFs or Macrophages, which showed real
# DE/pathway signal already and would be riskier reference choices).
# ----------------------------
adata2 = sc.read_h5ad(PROCESSED_DIR / "GSE176078_phase2_v2_annotated_corrected.h5ad", backed="r")

pairwise_comparisons = [("TNBC", "ER+"), ("HER2+", "ER+"), ("TNBC", "HER2+")]
all_sccoda_results_2 = {}

for group_a, group_b in pairwise_comparisons:
    comparison_name = f"{group_a}_vs_{group_b}"
    obs2_sub = adata2.obs[adata2.obs["subtype"].isin([group_a, group_b])].copy()
    obs2_sub["subtype"] = obs2_sub["subtype"].astype(str)  # drop unused categorical levels

    comp_adata2 = build_composition_anndata(obs2_sub, sample_col="orig.ident", condition_col="subtype")
    print(f"\n{comparison_name}: {comp_adata2.n_obs} samples x {comp_adata2.n_vars} cell types")

    model2 = scc_ana.CompositionalAnalysis(
        comp_adata2, formula="subtype", reference_cell_type="PVL"
    )
    result2 = model2.sample_hmc()

    print(f"=== GSE176078: {comparison_name} — scCODA results ===")
    result2.summary()

    effects2 = result2.effect_df
    effects2.to_csv(RESULTS_DIR / f"GSE176078_scCODA_{comparison_name}_effects.csv")
    all_sccoda_results_2[comparison_name] = result2

    credible2 = result2.credible_effects()
    print(f"\nCredible effects ({comparison_name}):")
    print(credible2)
    gc.collect()

print("\nGSE176078 scCODA complete")


TNBC_vs_ER+: 21 samples x 16 cell types
Zero counts encountered in data! Added a pseudocount of 0.5.


100%|██████████| 20000/20000 [03:47<00:00, 87.85it/s] 


MCMC sampling finished. (288.411 sec)
Acceptance rate: 56.8%
=== GSE176078: TNBC_vs_ER+ — scCODA results ===
Compositional Analysis summary:

Data: 21 samples, 16 cell types
Reference index: 12
Formula: subtype

Intercepts:
                        Final Parameter  Expected Sample
Cell Type                                               
B cells                          -1.050       121.965139
Basal epithelial                 -1.465        80.538494
CAFs                             -0.253       270.625313
CD8 T cells                      -0.121       308.812796
Cycling T cells                  -1.232       101.670303
Cycling epithelial               -0.823       153.045499
Endothelial cells                -0.037       335.873719
Epithelial (ambiguous)           -0.862       147.191617
Luminal epithelial                0.342       490.652066
Macrophages                       0.040       362.757749
Memory T cells                   -0.282       262.889885
NK cells                         -0

100%|██████████| 20000/20000 [03:06<00:00, 107.23it/s]


MCMC sampling finished. (236.422 sec)
Acceptance rate: 64.9%
=== GSE176078: HER2+_vs_ER+ — scCODA results ===
Compositional Analysis summary:

Data: 16 samples, 16 cell types
Reference index: 12
Formula: subtype

Intercepts:
                        Final Parameter  Expected Sample
Cell Type                                               
B cells                          -0.882       124.331051
Basal epithelial                 -1.490        67.690631
CAFs                             -0.253       233.211968
CD8 T cells                      -0.005       298.851793
Cycling T cells                  -1.298        82.018741
Cycling epithelial               -0.919       119.814867
Endothelial cells                 0.192       363.924990
Epithelial (ambiguous)           -1.004       110.051427
Luminal epithelial                0.495       492.723307
Macrophages                      -0.036       289.729513
Memory T cells                   -0.113       268.257616
NK cells                         -

100%|██████████| 20000/20000 [02:52<00:00, 115.62it/s]


MCMC sampling finished. (220.909 sec)
Acceptance rate: 46.8%
=== GSE176078: TNBC_vs_HER2+ — scCODA results ===
Compositional Analysis summary:

Data: 15 samples, 16 cell types
Reference index: 12
Formula: subtype

Intercepts:
                        Final Parameter  Expected Sample
Cell Type                                               
B cells                          -0.635       130.657292
Basal epithelial                 -1.335        64.882491
CAFs                              0.060       261.799202
CD8 T cells                       0.901       607.029753
Cycling T cells                  -0.917        98.551357
Cycling epithelial               -0.663       127.049631
Endothelial cells                -0.224       197.073707
Epithelial (ambiguous)           -0.512       147.758296
Luminal epithelial               -0.006       245.078313
Macrophages                       0.462       391.340414
Memory T cells                    0.959       643.278532
NK cells                         

In [8]:
# ----------------------------
# Cell 6 — Composition boxplots, with credible effects highlighted
# Shows the actual per-sample proportions (not just the model's summary
# statistic), with credible cell types marked — lets a reader see the
# real data behind the statistical claim, not just trust the model output.
# ----------------------------
import matplotlib.pyplot as plt
import numpy as np

def plot_composition_boxplot(obs_df, sample_col, condition_col, credible_types,
                               title, save_path, exclude_types=EXCLUDE_TYPES):
    df = obs_df[~obs_df["cell_type"].isin(exclude_types)].copy()
    counts = df.groupby([sample_col, "cell_type"], observed=True).size().unstack(fill_value=0)
    props = counts.div(counts.sum(axis=1), axis=0) * 100

    condition_lookup = df.drop_duplicates(sample_col).set_index(sample_col)[condition_col]
    props[condition_col] = condition_lookup.loc[props.index].values

    cell_types = [c for c in props.columns if c != condition_col]
    conditions = sorted(props[condition_col].unique())
    n_types = len(cell_types)

    fig, axes = plt.subplots(1, n_types, figsize=(n_types * 1.8, 5), sharey=False)
    if n_types == 1:
        axes = [axes]

    for ax, ct in zip(axes, cell_types):
        data_by_cond = [props.loc[props[condition_col] == c, ct].values for c in conditions]
        bp = ax.boxplot(data_by_cond, labels=conditions, patch_artist=True, widths=0.6)
        for patch in bp["boxes"]:
            patch.set_facecolor("lightgrey")
        for i, d in enumerate(data_by_cond):
            x = np.random.normal(i + 1, 0.04, size=len(d))
            ax.scatter(x, d, color="black", s=15, zorder=3, alpha=0.7)

        is_credible = ct in credible_types
        title_color = "firebrick" if is_credible else "black"
        title_weight = "bold" if is_credible else "normal"
        ax.set_title(f"{ct}{' *' if is_credible else ''}", fontsize=8,
                     color=title_color, fontweight=title_weight)
        ax.tick_params(axis="x", labelsize=7, rotation=45)
        ax.tick_params(axis="y", labelsize=7)

    fig.suptitle(f"{title}\n(* = credible effect, scCODA)", fontsize=11)
    fig.supylabel("Proportion (%)", fontsize=9)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close()

# GSE114725
credible_1 = credible1[credible1].index.get_level_values("Cell Type").tolist()
plot_composition_boxplot(
    obs1, "patient", "tissue", credible_1,
    "GSE114725 — Cell type composition, Tumour vs Normal",
    FIGURE_DIR / "GSE114725_composition_boxplot.png"
)

# GSE176078 — one figure per pairwise comparison
for comparison_name, result in all_sccoda_results_2.items():
    credible = result.credible_effects()
    credible_types = credible[credible].index.get_level_values("Cell Type").tolist()
    group_a, group_b = comparison_name.split("_vs_")
    obs2_sub = adata2.obs[adata2.obs["subtype"].isin([group_a, group_b])].copy()
    obs2_sub["subtype"] = obs2_sub["subtype"].astype(str)
    plot_composition_boxplot(
        obs2_sub, "orig.ident", "subtype", credible_types,
        f"GSE176078 — Cell type composition, {comparison_name.replace('_', ' ')}",
        FIGURE_DIR / f"GSE176078_composition_boxplot_{comparison_name}.png"
    )

print("Composition boxplots saved")

C:\Users\annam\AppData\Local\Temp\ipykernel_14708\783011735.py:29: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data_by_cond, labels=conditions, patch_artist=True, widths=0.6)
C:\Users\annam\AppData\Local\Temp\ipykernel_14708\783011735.py:29: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data_by_cond, labels=conditions, patch_artist=True, widths=0.6)
C:\Users\annam\AppData\Local\Temp\ipykernel_14708\783011735.py:29: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data_by_cond, labels=conditions, patch_artist=True, widths=0.6)
C:\Users\annam\AppData\Local\Temp\ipykernel_14

Composition boxplots saved
